# Kapitel 13: Feinabstimmung deines Modells

> "Das Glück ist das, was passiert, wenn Vorbereitung auf Gelegenheit trifft."
> — **Seneca**

---

## Was du lernen wirst

- Wann man Feinabstimmung anstelle besserer Prompts verwenden sollte
- Wie man Instruktions-/Antwort-Trainingsdaten vorbereitet
- Warum Loss-Maskierung das Training auf das Wesentliche fokussiert
- Wie LoRA mit Rang 8 für eine 768 × 768 grosse Projektion nur ein Achtundvierzigstel der Gewichte trainiert
- Auswertung des Modellverhaltens vor und nach der Feinabstimmung

---

## Setup

Zuerst installieren wir die benötigten Pakete und prüfen die GPU-Verfügbarkeit.

In [ ]:
# Getestete Paketversionen
!pip install -q torch==2.15.0 transformers==5.23.0 tqdm==4.70.1 numpy==2.3.5 matplotlib==3.10.7

In [ ]:
# ===== IMPORTS =====
import math
import json
import os
from dataclasses import dataclass
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer
from tqdm import tqdm

# GPU prüfen
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Verwendetes Gerät: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Speicher: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNUNG: Keine GPU erkannt. Training wird langsam sein.")
    print("Gehe zu Runtime > Change runtime type > GPU")

In [ ]:
# ===== REPRODUZIERBARKEIT =====
import random
import numpy as np

def set_seed(seed=42):
    """Setzt den Seed jeder Zufallsquelle (derselbe Helfer wie in Kapitel 12)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)


## 1. Modellkomponenten aus den Kapiteln 10-12

Zuerst holen wir das MiniGPT-Modell, das wir in den vorherigen Kapiteln erstellt haben.

In [ ]:
# ===== MULTI-HEAD ATTENTION (aus Kapitel 10) =====

class MultiHeadAttention(nn.Module):
    """Effiziente mehrköpfige Aufmerksamkeit (alle Köpfe zusammen verarbeitet)."""

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model muss durch num_heads teilbar sein"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        # bias=True entspricht GPT-2, damit wir später dessen vortrainierte Biases laden können
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)
        scores = scores / math.sqrt(d_k)

        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        attn_output = attn_weights @ V
        attn_output = attn_output.transpose(1, 2)
        attn_output = attn_output.reshape(batch, seq, d_model)

        output = self.out_proj(attn_output)
        return output, attn_weights

print("MultiHeadAttention definiert!")

In [ ]:
# ===== FEEDFORWARD-NETZWERK (aus Kapitel 10) =====

class FeedForward(nn.Module):
    """Positionsweises Feedforward-Netzwerk."""

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x, approximate="tanh")  # GPT-2s GELU-Variante
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print("FeedForward definiert!")

In [ ]:
# ===== TRANSFORMER-BLOCK (aus Kapitel 10) =====

class TransformerBlock(nn.Module):
    """Vollständiger Transformer-Block (Pre-Norm-Stil wie GPT-2)."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)
        return x, attn_weights

print("TransformerBlock definiert!")

In [ ]:
# ===== GPT-KONFIGURATION (aus Kapitel 11) =====
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """Konfiguration für das MiniGPT-Modell."""
    vocab_size: int = 50257
    max_seq_len: int = 1024
    embed_dim: int = 768
    num_heads: int = 12
    num_layers: int = 12
    d_ff: int = 3072
    dropout: float = 0.1

    def __post_init__(self):
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}) muss durch num_heads ({self.num_heads}) teilbar sein"

print("GPTConfig definiert!")

In [ ]:
# ===== MINIGPT-MODELL (aus Kapitel 11) =====

class MiniGPT(nn.Module):
    """Ein minimales GPT-Sprachmodell."""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # Einbettungen
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # Transformer-Blöcke
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # Finale Schichtnormalisierung und LM-Kopf
        self.ln_f = nn.LayerNorm(config.embed_dim)
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # Weight Tying
        self.lm_head.weight = self.token_embed.weight

        # Gewichte initialisieren
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        batch, seq = token_ids.shape
        device = token_ids.device

        tok_emb = self.token_embed(token_ids)
        positions = torch.arange(seq, device=device)
        pos_emb = self.pos_embed(positions)
        x = self.dropout(tok_emb + pos_emb)

        mask = torch.tril(torch.ones(seq, seq, device=device))

        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        if return_attention:
            return logits, attention_weights
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 do_sample=True, eos_token_id=None):
        """
        Tokens generieren, die input_ids fortsetzen (Prompt ist in der Ausgabe enthalten).

        Args:
            input_ids: Prompt-Token-IDs (batch, seq)
            max_new_tokens: Wie viele neue Tokens generiert werden
            temperature: Sampling-Temperatur (nur verwendet, wenn do_sample=True)
            do_sample: Aus der Verteilung samplen; False = gieriges argmax
            eos_token_id: Früh stoppen, wenn jede Sequenz dieses Token generiert

        Returns:
            Token-IDs (batch, seq + neue Tokens)
        """
        self.eval()

        # Positions-Einbettungen decken nur max_seq_len Positionen ab
        limit = self.config.max_seq_len
        if input_ids.shape[1] + max_new_tokens > limit:
            raise ValueError(
                f"Prompt ({input_ids.shape[1]} Tokens) + max_new_tokens "
                f"({max_new_tokens}) überschreitet den {limit}-Token-Kontext "
                "des Modells. Kürze den Prompt oder senke max_new_tokens.")
        for _ in range(max_new_tokens):
            logits = self(input_ids)
            next_logits = logits[:, -1, :]  # (batch, vocab_size)

            if do_sample:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logits.argmax(dim=-1, keepdim=True)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_token_id is not None and (next_token == eos_token_id).all():
                break

        return input_ids

print("MiniGPT-Klasse definiert!")


## 2. Das Kapitel-12-Basismodell holen (Checkpoint-Import)

Das Buch finetunt den **Kapitel-12-Shakespeare-Checkpoint**, und dieses Notebook auch. Baseline, LoRA-Training und Adapter-Reload nutzen **dieselben Basisgewichte**. Zwei ehrliche Wege:

- **Weg A (bevorzugt):** Kopiere deine eigene Datei `best_model.pt` aus Kapitel 12 in das Arbeitsverzeichnis dieses Notebooks. In Colab lädst du sie aus der Laufzeit von Kapitel 12 herunter und hier über die Seitenleiste **Dateien** wieder hoch. Verwende nur einen Checkpoint, den du selbst erstellt hast.
- **Weg B (frisches Colab, keine Datei):** Die Zelle unten führt Kapitel 12s Rezept als **1-Epochen-Minitraining** auf TinyShakespeare aus und speichert `best_model.pt`. Diese Basis ist schwächer als der 3-Epochen-Lauf des Buchs; die Losses weichen ab, aber jede Zelle nutzt diese eine konsistente Basis.

Eine zufällig initialisierte Basis ist ein *separates, klar gekennzeichnetes Experiment* am Ende, nie der Standard.

In [ ]:
# ===== KAPITEL-12-CHECKPOINT-IMPORT =====
# Jede Zelle unten (Baseline, LoRA-Training, Reload) nutzt DIESE Basis.
import os
import urllib.request

config = GPTConfig(
    vocab_size=50257, max_seq_len=128, embed_dim=256,
    num_heads=4, num_layers=4, d_ff=1024, dropout=0.1
)

# Der schnelle GPT-2-Tokenizer liefert die Zeichen-Offsets fürs Loss Masking.
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# Weg A: Alternativ kannst du deinen eigenen Checkpoint aus Kapitel 12 hier hochladen.
# from google.colab import files
# files.upload()  # best_model.pt auswählen

BASE_SOURCE = "Eigener Kapitel-12-Checkpoint"
if not os.path.exists("best_model.pt"):
    BASE_SOURCE = "Kapitel-12-Minitraining (eine Epoche)"
    # Weg B: Ohne Checkpoint trainieren wir eine Epoche nach dem Rezept aus Kapitel 12.
    # Chunks mit 512 Zeichen, Batch-Grösse 32, AdamW mit linearem Warmup.
    # Die Aufmerksamkeitsmaske kennzeichnet die Padding-Positionen.
    print("best_model.pt fehlt. Das Minitraining aus Kapitel 12 startet (eine Epoche).")
    from transformers import get_linear_schedule_with_warmup

    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        "shakespeare.txt")
    text = open("shakespeare.txt").read()

    def create_lm_labels(token_ids):
        inputs = token_ids[:, :-1]
        targets = token_ids[:, 1:].clone()
        return inputs, targets  # Die Schleife maskiert Padding anhand von attention_mask.

    class TextDataset(Dataset):
        def __init__(self, text, chunk_size=512):
            self.chunks = [text[i:i + chunk_size]
                           for i in range(0, len(text) - chunk_size + 1, chunk_size)]
        def __len__(self):
            return len(self.chunks)
        def __getitem__(self, idx):
            return self.chunks[idx]

    def ch12_collate(batch, tokenizer, max_length=128):
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt")
        return enc["input_ids"], enc["attention_mask"]

    ds = TextDataset(text, chunk_size=512)
    n_train = int(0.9 * len(ds))
    train_ds, val_ds = torch.utils.data.random_split(
        ds, [n_train, len(ds) - n_train],
        generator=torch.Generator().manual_seed(42))
    collate12 = partial(ch12_collate, tokenizer=tokenizer, max_length=128)
    train_loader12 = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate12)

    set_seed(42)
    model = MiniGPT(config).to(device)
    optimizer = AdamW(model.parameters(), lr=3e-4)
    total_steps12 = len(train_loader12)
    warmup_steps12 = min(100, max(1, total_steps12 // 10))
    print(f"Trainingsschritte: {total_steps12}, Warmup: {warmup_steps12}")
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps12,
        num_training_steps=total_steps12)

    model.train()
    total_loss = 0
    for input_ids, attention_mask in tqdm(train_loader12, desc="Minitraining aus Kapitel 12"):
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # Padding zählt nicht zum Verlust
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                               targets.reshape(-1), ignore_index=-100)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"Minitraining abgeschlossen, mittlerer Verlust: {total_loss / len(train_loader12):.4f}")
    # Nur Gewichte speichern, ohne Optimierer- oder Zufallszustand.
    torch.save({"model_state_dict": model.state_dict()}, "best_model.pt")
    print("best_model.pt gespeichert (eine Epoche; schwächere Basis als im Buchlauf mit drei Epochen).")

# Lade bei beiden Wegen denselben Checkpoint für alle folgenden Schritte.
# Kapitel 12 speichert auch NumPy-Zufallszustände. Erlaube dem sicheren Loader
# genau die benötigten Klassen.
import numpy as np
torch.serialization.add_safe_globals([np._core.multiarray._reconstruct,
                                      np.ndarray, np.dtype])
base_model = MiniGPT(config).to(device)
try:
    checkpoint = torch.load("best_model.pt", map_location=device, weights_only=True)
except Exception:
    # Ältere Checkpoints können weitere NumPy-Typen enthalten. Lade so nur
    # eine vertrauenswürdige Datei, die du in Kapitel 12 selbst erzeugt hast.
    checkpoint = torch.load("best_model.pt", map_location=device, weights_only=False)
base_model.load_state_dict(checkpoint["model_state_dict"])
base_model.eval()
print(f"Basismodell bereit: {sum(p.numel() for p in base_model.parameters()):,} Parameter")
print(f"Basisgewichte: {BASE_SOURCE}")

## 3. Feinabstimmungs-Datensatz vorbereiten

Wir erstellen dieselben deutschen Fragen, Präfixe und Antworten wie im Kapitel. Die Preisantwort steht in Kurzform: 10 Euro pro Monat für Einzelpersonen, 8 Euro pro Nutzer für Teams. Die englische Referenzdatei `data/faq_dataset.jsonl` im Begleitpaket wird nicht geladen. Mit Seed 42 bleiben fünf Kernformulierungen (20 Beispiele) fürs Testen übrig. Ihre vier Antworttexte kommen auch im Training vor: Wir testen neue Formulierungen bekannter Anliegen.

In [ ]:
# ===== DEUTSCHE FAQ: SPLIT NACH UMFORMULIERUNGSGRUPPEN =====
import json
import random

# Sieben handgeschriebene FAQ-Paare für unser fiktives Unternehmen
base_examples = [
    {"instruction": "Was macht TechStartup Inc?",
     "response": "TechStartup Inc entwickelt KI-gestützte Tools."},

    {"instruction": "Wann wurde TechStartup Inc gegründet?",
     "response": "TechStartup Inc wurde 2020 gegründet."},

    {"instruction": "Wo befindet sich TechStartup Inc?",
     "response": "TechStartup Inc befindet sich in Amsterdam."},

    {"instruction": "Was ist SmartScheduler?",
     "response": "SmartScheduler ist unser KI-Kalenderassistent."},

    {"instruction": "Wie viel kostet SmartScheduler?",
     "response": "10 Euro/Monat individuell, 8 Euro/Benutzer Teams."},

    {"instruction": "Wie setze ich mein Passwort zurück?",
     "response": "Klick auf 'Passwort vergessen' auf der Login-Seite."},

    {"instruction": "Kann ich meine Daten exportieren?",
     "response": "Ja, geh zu Einstellungen > Daten > Exportieren."},
]

# Umformulierungen pro Frage (gleiche Reihenfolge wie base_examples)
paraphrases = [
    ["Was macht TechStartup Inc?", "Was ist das Geschäft von TechStartup Inc?",
     "Erzähl mir, was TechStartup Inc herstellt.", "Was für ein Unternehmen ist TechStartup Inc?"],
    ["Wann wurde TechStartup Inc gegründet?", "In welchem Jahr startete TechStartup Inc?",
     "Wie alt ist TechStartup Inc?", "Wann fing TechStartup Inc an?"],
    ["Wo befindet sich TechStartup Inc?", "Wo hat TechStartup Inc seinen Sitz?",
     "In welcher Stadt ist TechStartup Inc?", "Wo sind die Büros von TechStartup Inc?"],
    ["Was ist SmartScheduler?", "Erzähl mir etwas über SmartScheduler.",
     "Was macht SmartScheduler?", "Beschreibe SmartScheduler."],
    ["Wie viel kostet SmartScheduler?", "Was ist der Preis von SmartScheduler?",
     "Was kostet SmartScheduler pro Monat?", "Wie viel kostet ein SmartScheduler-Abonnement?"],
    ["Wie setze ich mein Passwort zurück?", "Ich habe mein Passwort vergessen, was soll ich tun?",
     "Wie kann ich mein Passwort ändern?", "Wie läuft das Zurücksetzen des Passworts ab?"],
    ["Kann ich meine Daten exportieren?", "Wie exportiere ich meine Daten?",
     "Ist ein Datenexport möglich?", "Wo finde ich die Datenexport-Option?"],
]

def expand_faq(base_examples, paraphrases, seed=42):
    """Erweitere die FAQ-Paare deterministisch, mit Split nach Umformulierungs-GRUPPE.

    Jede Umformulierung plus ihre vier Höflichkeits-Präfix-Varianten bildet
    eine Gruppe (7 Fragen x 4 Umformulierungen = 28 Gruppen zu je 4). Wir
    mischen die GRUPPEN und halten 5 ganze Gruppen (20 Beispiele) zurück,
    BEVOR ein Train-/Eval-String in der anderen Aufteilung existiert, sodass
    keine Evaluierungs-Formulierung, mit Präfix oder ohne, je trainiert wurde.
    Training nimmt 80 der verbleibenden 92 Beispiele.
    """
    prefixes = ["", "Bitte sag mir: ", "Kurze Frage: ", "Könntest du mir sagen: "]
    groups = []
    for qid, (example, variants) in enumerate(zip(base_examples, paraphrases)):
        for paraphrase in variants:
            groups.append((qid, paraphrase,
                           [{"instruction": p + paraphrase,
                             "response": example["response"]} for p in prefixes]))
    rng = random.Random(seed)
    rng.shuffle(groups)

    # 5 ganze Gruppen zurückhalten, aber nie die LETZTE Gruppe einer Frage:
    # jeder Intent muss mindestens eine Formulierung im Training behalten.
    left = {qid: sum(1 for q, _, _ in groups if q == qid)
            for qid in range(len(base_examples))}
    eval_groups, train_groups = [], []
    for qid, paraphrase, examples in groups:
        if len(eval_groups) < 5 and left[qid] > 1:
            eval_groups.append((qid, paraphrase, examples))
        else:
            train_groups.append((qid, paraphrase, examples))
        left[qid] -= 1

    eval_data = [ex for _, _, g in eval_groups for ex in g]      # 5 x 4 = 20
    train_pool = [ex for _, _, g in train_groups for ex in g]    # 23 x 4 = 92
    rng.shuffle(train_pool)
    train_data = train_pool[:80]
    return train_data, eval_data

train_data, eval_data = expand_faq(base_examples, paraphrases)

# Assert 1: exakte Fragen-Strings sind disjunkt (der klassische Leakage-Check)
train_questions = {ex["instruction"] for ex in train_data}
assert all(ex["instruction"] not in train_questions for ex in eval_data)

# Assert 2: Umformulierungs-GRUPPEN sind disjunkt. Streife die höflichen
# Präfixe ab und prüfe, dass keine Eval-Formulierung in IRGENDEINER
# Präfix-Form im Training auftaucht.
def core_question(instruction):
    for p in ["Bitte sag mir: ", "Kurze Frage: ", "Könntest du mir sagen: "]:
        if instruction.startswith(p):
            return instruction[len(p):]
    return instruction

train_cores = {core_question(q) for q in train_questions}
assert all(core_question(ex["instruction"]) not in train_cores for ex in eval_data)

# Prüfe, welche Antworten der Holdout tatsächlich abdeckt.
train_intents = {ex["response"] for ex in train_data}
eval_intents = {ex["response"] for ex in eval_data}
assert eval_intents <= train_intents
print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")
print(f"Anliegen im Training: {len(train_intents)}/7")
print(f"Anliegen im Holdout: {len(eval_intents)}/7; "
      f"davon im Training: {len(eval_intents & train_intents)}")
print(f"Eval-Formulierungen im Training gesehen (Präfixe entfernt): "
      f"{sum(core_question(ex['instruction']) in train_cores for ex in eval_data)}/20")
# Ausgabe: Train: 80, Eval: 20
# Ausgabe: Anliegen im Training: 7/7
# Ausgabe: Anliegen im Holdout: 4/7; davon im Training: 4
# Ausgabe: Eval-Formulierungen im Training gesehen (Präfixe entfernt): 0/20

def save_as_jsonl(examples, filepath):
    """Speichere Beispiele im JSONL-Format (ein JSON-Objekt pro Zeile)."""
    with open(filepath, 'w') as f:
        for example in examples:
            f.write(json.dumps(example) + '\n')

def load_jsonl(filepath):
    """Lade Beispiele aus dem JSONL-Format."""
    examples = []
    with open(filepath, 'r') as f:
        for line in f:
            examples.append(json.loads(line))
    return examples

# Speichere unseren Datensatz (die englische Referenzdatei im Begleitpaket ist data/faq_dataset.jsonl)
save_as_jsonl(train_data + eval_data, 'faq_dataset.jsonl')


In [ ]:
# ===== FORMATIERUNG + BATCHES (aus dem Buch) =====
# Definiere spezielle Tokens
INST_START = "[INST]"
INST_END = "[/INST]"

def build_prompt(instruction):
    """Das EINE Prompt-Format, das Training und Generierung teilen.

    Endet direkt nach [/INST], ohne abschliessendes Leerzeichen: Ein
    abschliessendes Leerzeichen kann allein anders tokenisieren als
    zusammen mit dem ersten Antwortwort (" Tech"), wodurch die Inferenz
    dem Modell einen Präfix füttern würde, den es im Training nie sah.
    """
    return f"{INST_START} {instruction} {INST_END}"

def format_example(instruction, response, tokenizer):
    """
    Formatiere ein Instruktions-/Antwortpaar für das Training.

    Braucht einen schnellen Tokenizer (GPT2TokenizerFast, den unser Setup lädt):
    Nur schnelle Tokenizer melden Zeichen-Offsets für jedes Token.

    Rückgabe:
        input_ids: Vollständige tokenisierte Sequenz
        labels: Wie input_ids, aber mit -100 für Instruktions-Tokens
    """
    # Format: [INST] Instruktion [/INST] Antwort + End-of-Text.
    # Das EOS-Token bringt dem Modell bei, wo Antworten ENDEN, damit die
    # Generierung dort stoppen kann, statt nach der Antwort weiterzuschwafeln.
    prompt = build_prompt(instruction)
    full_text = prompt + " " + response + tokenizer.eos_token

    # Tokenisieren und die Zeichenspanne jedes Tokens in full_text anfordern
    enc = tokenizer(full_text, return_offsets_mapping=True)
    input_ids = torch.tensor(enc["input_ids"])

    # Finde heraus, wo die Antwort beginnt: das erste Token, dessen
    # Zeichenspanne vollständig hinter dem Prompt liegt. BPE klebt das
    # hinzugefügte Leerzeichen an das erste Antwortwort (" Tech"), dieses
    # Token gehört also zur Antwort; blosses Zählen der Prompt-Tokens
    # würde es maskieren.
    response_start = next(i for i, (_, end) in enumerate(enc["offset_mapping"])
                          if end > len(prompt))

    # Erstelle Labels: -100 für Instruktions-Tokens (vom Loss ignoriert)
    labels = input_ids.clone()
    labels[:response_start] = -100  # Maskiere Instruktions-Tokens

    return input_ids, labels


import torch
from functools import partial
from torch.utils.data import DataLoader

PAD_ID = tokenizer.eos_token_id  # GPT-2 paddet mit End-of-Text

def collate_fn(examples, tokenizer, max_length=128):
    """Instruktions-/Antwort-Paare zu gepaddeten input_ids + labels tokenisieren."""
    batch_ids, batch_labels = [], []
    for ex in examples:
        ids, labels = format_example(ex['instruction'], ex['response'], tokenizer)
        ids, labels = ids[:max_length], labels[:max_length]
        pad = max_length - len(ids)
        ids = torch.cat([ids, torch.full((pad,), PAD_ID, dtype=torch.long)])
        labels = torch.cat([labels, torch.full((pad,), -100, dtype=torch.long)])
        batch_ids.append(ids)
        batch_labels.append(labels)
    return {'input_ids': torch.stack(batch_ids),
            'labels': torch.stack(batch_labels)}

collate = partial(collate_fn, tokenizer=tokenizer)
train_loader = DataLoader(train_data, batch_size=8, shuffle=True, collate_fn=collate)


# Einen Batch prüfen
batch = next(iter(train_loader))
print(f"Input-IDs-Form: {batch['input_ids'].shape}")
print(f"Labels-Form: {batch['labels'].shape}")
print(f"Maskierte Positionen im ersten Beispiel: {(batch['labels'][0] == -100).sum().item()}")


## 4. Baseline-Auswertung (VOR der Feinabstimmung)

Schauen wir uns an, wie das Basismodell auf unsere FAQ-Prompts reagiert.

In [ ]:
@torch.no_grad()
def generate_response(model, prompt, tokenizer, max_new_tokens=50, temperature=0.8):
    """Eine Antwort auf einen Instruktions-Prompt generieren."""
    model.eval()
    full_prompt = build_prompt(prompt)
    input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(device)
    
    output_ids = model.generate(
        input_ids, 
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True
    )
    
    response = tokenizer.decode(output_ids[0][len(input_ids[0]):])
    return response.strip()

# Testen mit FAQ-Prompts
test_prompts = [
    "Was macht TechStartup Inc?",
    "Wie setze ich mein Passwort zurück?",
    "Was ist SmartScheduler?"
]

print(f"VOR DER FEINABSTIMMUNG ({BASE_SOURCE}):")
print("="*60)
for prompt in test_prompts:
    response = generate_response(base_model, prompt, tokenizer)
    print(f"\nF: {prompt}")
    print(f"A: {response[:100]}...")
print("\n(Shakespeare-Kauderwelsch - die Kapitel-12-Basis weiss nichts über")
print("TechStartup Inc, und ein 16M-Modell von Grund auf hat nie fliessendes")
print("Englisch gelernt. Ehrliche Erwartungen!)")


## 5. LoRA-Komponenten definieren

Erstelle die LoRA-Adapterschicht, die trainierbare Parameter hinzufügt, ohne die Basisgewichte zu verändern.

### Warum LoRA statt vollständiger Feinabstimmung?

Die vollständige Feinabstimmung hat ein Problem: **katastrophales Vergessen**. Wenn du alle Gewichte für eine neue Aufgabe aktualisierst, kann das Modell "vergessen", was es während des Vortrainings gelernt hat. LoRA reduziert dies elegant, indem es die Basisgewichte *eingefroren* hält und kleine trainierbare Matrizen *daneben* hinzufügt.

Denk dabei an eine Lesebrille: Deine Augen (Basismodell) bleiben gleich, aber die Brille (LoRA) fügt eine kleine Korrektur für bestimmte Aufgaben hinzu.

In [ ]:
# ===== LORA-LINEAR-SCHICHT (Paper-Konvention: A runter, B hoch) =====
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    """
    Eine lineare Schicht mit Low-Rank Adaptation (LoRA).

    Denk dabei an eine Lesebrille für deine Augen:
    - Deine Augen (Basisschicht) bleiben gleich
    - Die Brille (LoRA) fügt eine kleine Korrektur hinzu
    - Zusammen funktionieren sie besser als die Augen allein

    Mathematik: output = base(x) + (x @ A @ B) * scale
    """

    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()

        # Die ursprüngliche Schicht (wir frieren diese ein); Bias entspricht GPT-2s qkv_proj
        self.base = nn.Linear(in_features, out_features)

        # LoRA-Matrizen (Paper-Konvention: A runter, B hoch)
        # A: (in_features, rank) — "Down"-Projektion (komprimieren)
        # B: (rank, out_features) — "Up"-Projektion (erweitern)
        self.lora_A = nn.Parameter(torch.zeros(in_features, rank))
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))

        # A mit kleinen Zufallswerten initialisieren (Kaiming), B mit Nullen.
        # Über die transponierte Ansicht initialisieren, damit Kaiming
        # (rank, in_features) sieht, die Gewichtsform von nn.Linear(in_features, rank):
        # fan_in wird so zu in_features. In-place auf die Ansicht zu schreiben
        # aktualisiert lora_A selbst.
        nn.init.kaiming_uniform_(self.lora_A.T, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

        # Skalierungsfaktor
        self.scale = alpha / rank

    def freeze_base(self):
        """Die Basisschicht einfrieren, sodass nur die LoRA-Matrizen trainiert werden."""
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False

    def forward(self, x):
        # x-Form: (batch, sequence, in_features)

        # Ursprüngliche Ausgabe von eingefrorenen Gewichten
        base_output = self.base(x)  # (batch, seq, out_features)

        # LoRA-Pfad: x @ A @ B, skaliert
        # x @ A: (batch, seq, in_features) @ (in_features, rank)
        #      = (batch, seq, rank)  — komprimiert!
        # ... @ B: (batch, seq, rank) @ (rank, out_features)
        #        = (batch, seq, out_features)  — wieder erweitert
        lora_output = (x @ self.lora_A @ self.lora_B) * self.scale

        return base_output + lora_output

print("LoRALinear definiert!")

# Parametereinsparungen demonstrieren
in_features, out_features, rank = 256, 256, 8
full_params = in_features * out_features
lora_params = in_features * rank + rank * out_features

print(f"\nParametervergleich (256x256-Schicht):")
print(f"  Vollständige Feinabstimmung: {full_params:,} Parameter")
print(f"  LoRA (rank=8):               {lora_params:,} Parameter")
print(f"  Einsparungen:                {full_params/lora_params:.1f}x weniger!")


## 6. LoRA auf Modell anwenden

Attention-Projektionen durch LoRA-Versionen ersetzen und das Basismodell einfrieren.

In [ ]:
def add_lora_to_model(model, rank=8, alpha=16):
    """
    LoRA zu den Attention-Projektionen hinzufügen.

    Wir zielen auf qkv_proj, die kombinierte Query/Key/Value-Projektion.
    Der Adapter lernt gemeinsame Korrekturen für Q, K und V.
    (PEFT kann auch q/v separat adressieren.)
    """
    # Schritt 1: Zuerst ALLE Parameter einfrieren
    for param in model.parameters():
        param.requires_grad = False

    # Schritt 2: qkv_proj in jedem Transformer-Block ersetzen
    # Wir iterieren explizit durch die Blöcke, die wir zuvor erstellt haben
    for block in model.blocks:
        # Die QKV-Projektion der Attention-Schicht holen
        original_qkv = block.attn.qkv_proj
        in_features = original_qkv.in_features
        out_features = original_qkv.out_features

        # LoRA-erweiterten Ersatz auf demselben Gerät wie das Modell erstellen
        lora_qkv = LoRALinear(in_features, out_features, rank, alpha)
        lora_qkv = lora_qkv.to(original_qkv.weight.device)

        # Vortrainierte Gewichte (und Bias!) in die Basisschicht kopieren
        lora_qkv.base.weight.data = original_qkv.weight.data.clone()
        lora_qkv.base.bias.data = original_qkv.bias.data.clone()

        # Basis einfrieren (LoRA-Matrizen bleiben trainierbar)
        lora_qkv.freeze_base()
        if lora_qkv.base.bias is not None:
            lora_qkv.base.bias.requires_grad = False

        # Schicht ersetzen
        block.attn.qkv_proj = lora_qkv

    return model

# Dieselbe Basis wie im Buch: der oben importierte Kapitel-12-Checkpoint.
# Baseline, Training und Reload teilen diese Gewichte, damit die
# VORHER/NACHHER-Vergleiche ehrlich bleiben. (Bei Weg-B-Minitraining
# weichen die Losses von den 3-Epochen-Zahlen des Buchs ab.)
set_seed(42)  # VOR Adapter-Init und Batch-Shuffling (nach dem Training zu seeden reproduziert nichts)
model = MiniGPT(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model = add_lora_to_model(model, rank=8, alpha=16)

# Parameter zählen
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Gesamtparameter:      {total:,}")
print(f"Trainierbar (LoRA):   {trainable:,}")
print(f"Trainierbarer Prozentsatz: {100*trainable/total:.2f}%")

# Zeigen, was trainierbar ist
print("\nTrainierbare Parameter:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

## 7. Feinabstimmungs-Loop

Trainieren mit dem 5-Schritte-Rezept und maskiertem Loss.

In [ ]:
import torch.nn.functional as F

def compute_masked_loss(logits, labels):
    """
    Cross-Entropy-Loss berechnen, Positionen ignorieren, wo labels == -100.
    
    Das ist wie nur den Antwortteil einer Prüfung zu benoten,
    nicht die Frage, die vom Prompt kopiert wurde.
    """
    # Für Next-Token-Vorhersage verschieben
    # .contiguous() stellt sicher, dass Speicher sequenziell für .view() angeordnet ist
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    
    # Cross-Entropy mit ignore_index=-100
    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100  # PyTorch ignoriert diese Positionen automatisch!
    )
    
    return loss


def finetune_epoch(model, dataloader, optimizer, device):
    """
    Feinabstimmung für eine Epoche mit dem 5-Schritte-Rezept.
    """
    model.train()
    total_loss = 0
    
    progress = tqdm(dataloader, desc="Feinabstimmung")
    for batch in progress:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        
        # ===== DAS 5-SCHRITTE-REZEPT =====
        
        # Schritt 1: Gradienten nullen
        optimizer.zero_grad()
        
        # Schritt 2: Vorwärtsdurchlauf
        logits = model(input_ids)
        
        # Schritt 3: MASKIERTEN Loss berechnen
        loss = compute_masked_loss(logits, labels)
        
        # Schritt 4: Rückwärtsdurchlauf
        loss.backward()
        
        # Schritt 5: Gewichte aktualisieren (nur LoRA!)
        optimizer.step()
        
        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")
    
    return total_loss / len(dataloader)

print("Trainingsfunktionen definiert!")

In [ ]:
# ===== TRAINING =====

# Hyperparameter
num_epochs = 3
learning_rate = 1e-3  # LoRA-Adapter vertragen eine höhere Lernrate als vollständige Feinabstimmung

# Optimizer (nur trainierbare Parameter)
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=learning_rate  # Wie im Buch: AdamW mit seinen Standardwerten.
)

# Trainings-Loop
train_losses = []

print("Starte Feinabstimmung...")
print(f"Training mit {len(train_data)} Beispielen für {num_epochs} Epochen\n")

for epoch in range(num_epochs):
    print(f"\nEpoche {epoch + 1}/{num_epochs}")
    print("-" * 40)
    
    loss = finetune_epoch(model, train_loader, optimizer, device)
    train_losses.append(loss)
    
    print(f"Durchschnittlicher Loss: {loss:.4f}")

print("\nFeinabstimmung abgeschlossen!")
print("Hinweis: Mit einem von Grund auf trainierten 16M-Basismodell sind Kauderwelsch-")
print("Generierungen zu erwarten; der Gewinn ist der Loss-Abfall und die Adapter-Mechanik.")
print("Für fliessende Antworten starte von vortrainierten GPT-2-Gewichten (Loader aus Kapitel 11).")

## 8. Trainingsfortschritt plotten

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, 'b-o', linewidth=2, markersize=8)
plt.xlabel('Epoche', fontsize=12)
plt.ylabel('Trainings-Loss', fontsize=12)
plt.title('Feinabstimmungs-Loss-Kurve', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(range(1, len(train_losses) + 1))
plt.show()

print(f"\nLoss sank von {train_losses[0]:.4f} auf {train_losses[-1]:.4f}")
print(f"Verbesserung: {(1 - train_losses[-1]/train_losses[0])*100:.1f}%")

## 9. Auswertung (NACH der Feinabstimmung)

Der ehrliche Teil: messen, was sich verändert hat. Mit unserem von Grund auf trainierten 16M-Basismodell bleiben die Generierungen VORHER und NACHHER Kauderwelsch (es hat nie fliessendes Englisch gelernt). Prüfe deshalb neben dem Trainingsverlust den Antwort-Loss auf neuen Formulierungen bekannter Anliegen. Vergleiche deinen eigenen Checkpoint vor und nach dem Training: Sinkt dieser Loss, auch wenn sich an der exakten Übereinstimmung nichts ändert?

In [ ]:
def evaluate_model(model, test_examples, tokenizer, device):
    """
    Evaluiere feinabgestimmtes Modell auf Testbeispielen.
    """
    model.eval()
    results = []

    for example in test_examples:
        instruction = example['instruction']
        expected = example['response']

        # Generiere aus DEMSELBEN Präfix-Format, das das Training verwendet hat
        prompt = build_prompt(instruction)
        input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

        # Holdout-Loss: Zwinge die KORREKTE Antwort per Teacher Forcing
        # durch das Modell und bewerte nur Antwort-Tokens (derselbe
        # maskierte Loss wie im Training, aber auf Frageformulierungen, die die
        # Adapter nie gesehen haben)
        full_ids, full_labels = format_example(instruction, expected, tokenizer)
        with torch.no_grad():
            logits = model(full_ids.unsqueeze(0).to(device))
            resp_loss = compute_masked_loss(logits, full_labels.unsqueeze(0).to(device))

        output = model.generate(
            input_ids,
            max_new_tokens=50,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )

        # Dekodiere nur die neuen Tokens und lasse den EOS-Terminator selbst weg
        new_tokens = output[0][input_ids.shape[1]:].tolist()
        if tokenizer.eos_token_id in new_tokens:
            new_tokens = new_tokens[:new_tokens.index(tokenizer.eos_token_id)]
        generated = tokenizer.decode(new_tokens).strip()

        # Exakte Übereinstimmung
        is_exact = generated.lower() == expected.lower()

        # Wortüberlappung
        gen_words = set(generated.lower().split())
        exp_words = set(expected.lower().split())
        overlap = len(gen_words & exp_words) / max(len(exp_words), 1)

        results.append({
            'instruction': instruction,
            'expected': expected,
            'generated': generated,
            'exact_match': is_exact,
            'word_overlap': overlap,
            'response_loss': resp_loss.item()
        })

    # Zusammenfassung
    accuracy = sum(r['exact_match'] for r in results) / len(results)
    avg_overlap = sum(r['word_overlap'] for r in results) / len(results)

    return {
        'exact_match_accuracy': accuracy,
        'average_word_overlap': avg_overlap,
        'avg_response_loss': sum(r['response_loss'] for r in results) / len(results),
        'detailed_results': results
    }

# Auf dem zurückgehaltenen Split auswerten
print("Auswertung auf Testset...")
results = evaluate_model(model, eval_data, tokenizer, device)

print(f"\n===== AUSWERTUNGSERGEBNISSE =====")
print(f"Exakte Übereinstimmungsgenauigkeit: {results['exact_match_accuracy']*100:.1f}%")
print(f"Durchschnittliche Wortüberlappung: {results['average_word_overlap']*100:.1f}%")
print("(Nahe 0% ist mit einem von Grund auf trainierten Basismodell zu erwarten; siehe Hinweis oben.)")
print(f"Holdout-Antwort-Loss: {results['avg_response_loss']:.2f}")


In [ ]:
# Detaillierte Ergebnisse anzeigen
print("\n===== DETAILLIERTE TESTERGEBNISSE =====")
for r in results['detailed_results']:
    status = "✓" if r['exact_match'] else "○"
    print(f"\n{status} F: {r['instruction']}")
    print(f"  Erwartet:  {r['expected']}")
    print(f"  Generiert: {r['generated'][:100]}..." if len(r['generated']) > 100 else f"  Generiert: {r['generated']}")
    print(f"  Überlappung: {r['word_overlap']*100:.0f}%")

In [ ]:
# ===== VORHER vs. NACHHER VERGLEICH =====

# Basismodell zum Vergleich: exakte Kopie der Basisgewichte (unangepasst)
base_model_fresh = MiniGPT(config).to(device)
base_model_fresh.load_state_dict(base_model.state_dict())

print("\n" + "="*70)
print("VORHER vs. NACHHER Feinabstimmung")
print("="*70)

baseline_results = evaluate_model(base_model_fresh, eval_data, tokenizer, device)
for label, metrics in [("VORHER", baseline_results), ("NACHHER", results)]:
    print(f"{label}: exakte Übereinstimmung {metrics['exact_match_accuracy']:.0%}, "
          f"Wortüberlappung {metrics['average_word_overlap']:.2f}, "
          f"Antwort-Loss {metrics['avg_response_loss']:.2f}")

comparison_prompts = ["Was macht TechStartup Inc?", "Wie setze ich mein Passwort zurück?"]

set_seed(42)  # erneut seeden, damit der gesampelte Vergleich reproduzierbar ist
for prompt in comparison_prompts:
    full_prompt = build_prompt(prompt)
    input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        base_output = base_model_fresh.generate(
            input_ids, max_new_tokens=50,
            eos_token_id=tokenizer.eos_token_id)
        ft_output = model.generate(
            input_ids, max_new_tokens=50,
            eos_token_id=tokenizer.eos_token_id)

    before = tokenizer.decode(base_output[0][len(input_ids[0]):])
    after = tokenizer.decode(ft_output[0][len(input_ids[0]):])

    print(f"\nF: {prompt}")
    print(f"  VORHER: {before.strip()}")
    print(f"  NACHHER: {after.strip()}")
    print("-"*70)

print("Vergleiche beide Holdout-Verluste: Eine höhere Wahrscheinlichkeit für die")
print("bekannten Antworten ist nicht dasselbe wie eine verständliche generierte Antwort.")


## 10. LoRA-Gewichte speichern und laden

In [ ]:
def save_lora_weights(model, filepath):
    """Nur die LoRA-Parameter speichern (winzige Datei!)."""
    lora_state_dict = {
        name: param for name, param in model.state_dict().items()
        if 'lora_' in name
    }
    torch.save(lora_state_dict, filepath)

    # Dateigrösse anzeigen
    import os
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"LoRA-Gewichte gespeichert: {filepath} ({size_mb:.2f} MB)")



# LoRA-Gewichte speichern
save_lora_weights(model, 'techstartup_lora.pt')

# Mit vollständiger Modellgrösse vergleichen
torch.save(model.state_dict(), 'full_model.pt')
full_size = os.path.getsize('full_model.pt') / (1024 * 1024)
lora_size = os.path.getsize('techstartup_lora.pt') / (1024 * 1024)
print(f"Vollständiges Modell gespeichert: full_model.pt ({full_size:.1f} MB)")
print(f"\nLoRA ist {full_size/lora_size:.0f}x kleiner als das vollständige Modell!")


In [ ]:
def load_lora_weights(model, filepath):
    """LoRA-Gewichte in ein Modell laden, das bereits LoRA-Schichten hat."""
    lora_state_dict = torch.load(filepath, weights_only=True)
    model.load_state_dict(lora_state_dict, strict=False)
    print(f"LoRA-Gewichte geladen von: {filepath}")



# Laden demonstrieren
print("\n===== Laden/Speichern testen =====")
# Der Adapter ist nur auf genau der Basis sinnvoll, auf der er trainiert wurde:
# zuerst best_model.pt laden, dann die LoRA-Struktur, dann den Adapter.
new_model = MiniGPT(config).to(device)
new_model.load_state_dict(checkpoint["model_state_dict"])  # Basis zuerst!
new_model = add_lora_to_model(new_model, rank=8, alpha=16)

before_load = generate_response(new_model, "Was macht TechStartup Inc?", tokenizer)
print(f"Vor dem Laden von LoRA: {before_load[:60]}...")

load_lora_weights(new_model, 'techstartup_lora.pt')

after_load = generate_response(new_model, "Was macht TechStartup Inc?", tokenizer)
print(f"Nach dem Laden von LoRA: {after_load[:60]}...")
# Plausibilitätsprüfung: neu geladenes Modell muss dieselben Logits liefern
probe = tokenizer.encode("[INST] Was ist SmartScheduler? [/INST]",
                         return_tensors="pt").to(device)
model.eval(); new_model.eval()
with torch.no_grad():
    torch.testing.assert_close(new_model(probe), model(probe), atol=1e-5, rtol=0)
print("Reload-Paritätsprüfung bestanden")
print("\nLoRA-Adapter erfolgreich gespeichert und geladen!")


# ===== MERGE: LoRA in die Basis falten und die Wrapper ENTFERNEN =====
def merge_lora_into_base(model):
    """
    LoRA in die Basisgewichte falten und die Adapter-Wrapper ENTFERNEN,
    damit das gemergte Modell wieder ein normales MiniGPT ist (permanent).
    """
    for block in model.blocks:
        lora = block.attn.qkv_proj
        if isinstance(lora, LoRALinear):
            # Korrektur in das Basisgewicht falten: W + (A @ B * scale).T
            delta = lora.lora_A @ lora.lora_B * lora.scale
            lora.base.weight.data += delta.T  # Transponieren für die Linear-Konvention
            # Wrapper gegen die normale gemergte Schicht tauschen
            block.attn.qkv_proj = lora.base

    print("LoRA-Gewichte in das Basismodell gemergt")

# Mergen und prüfen: gleiche Logits, Adapter-Parameter weg.
# HINWEIS: Mergen ist permanent; führe diese Zelle erst nach den Vergleichen oben aus.
probe = tokenizer.encode("[INST] Was ist SmartScheduler? [/INST]",
                         return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    adapter_logits = model(probe)

merge_lora_into_base(model)

with torch.no_grad():
    merged_logits = model(probe)
torch.testing.assert_close(merged_logits, adapter_logits, atol=1e-5, rtol=0)

total = sum(p.numel() for p in model.parameters())
print(f"Parameter des gemergten Modells: {total:,} (Adapter entfernt)")
assert not any(isinstance(m, LoRALinear) for m in model.modules())
print("Merge-Paritätsprüfung bestanden")
# Warum transponieren? forward berechnet x @ A @ B, das Update ist (in, out);
# nn.Linear speichert Gewichte als (out, in), daher delta.T. Den Wrapper zu ersetzen
# (statt die Adapter nur zu nullen) entfernt die zwei toten Adapter-Matmuls.


## Zusammenfassung

**Was wir erstellt haben:**

1. **Instruktions-/Antwort-Datensatz** mit Loss-Maskierung
2. **LoRA-Adapter**, die <1% der Parameter trainieren
3. **Feinabstimmungs-Loop** mit dem 5-Schritte-Rezept
4. **Auswertung** des Vorher-/Nachher-Verhaltens (ehrlich: der Loss fällt, der Text bleibt Kauderwelsch bei einer von Grund auf trainierten Basis)
5. **Adapter-Speichern/Laden** für Deployment

**Wichtige Erkenntnisse:**

- Loss-Maskierung fokussiert das Training auf das Wesentliche (Antworten, nicht Instruktionen)
- Die Einsparung hängt von den Schichtdimensionen und vom Rang ab. Zähle die trainierbaren Gewichte deines eigenen Modells.
- Das Einfrieren der Basisgewichte reduziert **katastrophales Vergessen** (Verlust von vortrainiertem Wissen) deutlich
- Die Qualität der Daten ist wichtiger als die Quantität für die Feinabstimmung
- Adapter sind winzig und können für verschiedene Aufgaben ausgetauscht werden

**Nächstes:** Kapitel 14 wird Prompt Engineering erkunden - bessere Ausgaben erhalten, ohne die Modellgewichte überhaupt zu ändern!

## Optionales Experiment: Zufallsbasis-Variante (klar gekennzeichnet)

Der Hauptablauf oben nutzt überall den Kapitel-12-Checkpoint. Dieses Experiment wiederholt das LoRA-Rezept auf einer **zufällig initialisierten Basis**, um zu zeigen, warum der Checkpoint zählt. Es ist NICHT der Lauf des Buchs: Baseline, Training und Reload teilen hier dieselben Zufallsgewichte, getrennt vom Checkpoint-Ablauf. Erwarte einen anderen, meist höheren Anfangsverlust.

In [ ]:
# ===== ZUFALLSBASIS-VARIANTE (gekennzeichnetes Experiment, NICHT der Buch-Lauf) =====
set_seed(42)
rand_base = MiniGPT(config).to(device)  # zufällig initialisiert, ohne Checkpoint
rand_model = MiniGPT(config).to(device)
rand_model.load_state_dict(rand_base.state_dict())  # dieselben Zufallsgewichte
rand_model = add_lora_to_model(rand_model, rank=8, alpha=16)

rand_optimizer = AdamW([p for p in rand_model.parameters() if p.requires_grad],
                       lr=1e-3, weight_decay=0.01)
rand_losses = []
for epoch in range(3):
    loss = finetune_epoch(rand_model, train_loader, rand_optimizer, device)
    rand_losses.append(loss)
    print(f"[Zufallsbasis] Epoche {epoch+1}: maskierter Verlust {loss:.4f}")

print(f"\nVerluste mit Checkpoint-Basis: {[f'{l:.4f}' for l in train_losses]}")
print(f"Verluste mit Zufallsbasis:     {[f'{l:.4f}' for l in rand_losses]}")
print("Gleiches Rezept, andere Basis: Die Verluste hängen von den Startgewichten ab.")
print("Lade den Adapter deshalb immer auf die Basis, auf der er trainiert wurde.")

## Übungen

### Übung 1: Verschiedene LoRA-Ränge

Probier rank=4 und rank=16 aus. Wie beeinflusst es Training und Ergebnisse?

In [ ]:
# DEIN CODE HIER
# 1. Modell mit rank=4 erstellen
# 2. Für 3 Epochen trainieren
# 3. Loss und Auswertung mit rank=8 vergleichen

### Übung 2: Eigene FAQ erstellen

Erstelle einen Datensatz über ein Thema, das du kennst (deine Schule, Hobby, etc.).

In [ ]:
# DEIN CODE HIER
# 1. 30+ Instruktions-/Antwort-Paare erstellen
# 2. Modell feinabstimmen
# 3. Mit eigenen Fragen testen

### Übung 3: Loss-Maskierungs-Ablation

Was passiert, wenn wir die Instruktions-Tokens NICHT maskieren?

In [ ]:
# DEIN CODE HIER
# 1. Dataset modifizieren, um Instruktions-Tokens NICHT zu maskieren (alle labels = tatsächliche Tokens)
# 2. Modell trainieren
# 3. Ergebnisse mit maskierter Version vergleichen
# 4. Was beobachtest du?